In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


url = "https://old.reddit.com/r/Espana/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

print("Conectando a Reddit ")
response = requests.get(url, headers=headers, timeout=10)
lista_reddit = []

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    posts_html = soup.select("div.thing")
    
    for post in posts_html:
        bloque_titulo = post.select_one("p.title a.title")
        if not bloque_titulo: 
            continue
            
        titulo = bloque_titulo.get_text().strip()
        enlace = bloque_titulo["href"]
        
        # Corregir enlaces internos (si es que los hubiera)
        if enlace.startswith("/r/"):
            enlace = "https://old.reddit.com" + enlace
            
        # Extraer los votos de cada publicaion
        bloque_votos = post.select_one("div.score.unvoted")
        votos_texto = bloque_votos.get_text().strip() if bloque_votos else "0"
        
      
        lista_reddit.append({
            "Título": titulo,
            "Enlace": enlace,
            "Votos_Texto": votos_texto
        })


# PROCESAMIENTO CON PANDAS 

# Verificamos que la lista no esté vacía 
if len(lista_reddit) > 0:
    df_reddit = pd.DataFrame(lista_reddit)
    
    # Transformamos el texto de los votos a números puros 
    df_reddit["Votos"] = pd.to_numeric(df_reddit["Votos_Texto"], errors='coerce').fillna(0).astype(int)
    
    # Filtro de palabras clave de desinformación
    palabras_alarma = ["bulo", "falso", "estafa", "fraude", "urgente", "alerta", "clickbait","impactante", "increíble", "no lo vas a creer", "escándalo",
        "urgente", "alerta", "shock", "viral", "polémica", "explosivo"]
    
    patron_busqueda = "|".join(palabras_alarma)
    
    df_reddit["¿Es_Bulo?"] = df_reddit["Título"].str.lower().str.contains(patron_busqueda, na=False)
    
    # Separamos las tablas de resultados
    validas = df_reddit[df_reddit["¿Es_Bulo?"] == False]
    falsas = df_reddit[df_reddit["¿Es_Bulo?"] == True]
    
    # Imprimir resumen de resultados
    print("\n--- REPORTE DE EXTRACCIÓN ---")
    print(f"Total de noticias evaluadas: {len(df_reddit)}")
    print(f"Noticias normales validadas: {len(validas)}")
    print(f"Alertas de información falsa encontradas: {len(falsas)}")
    
    if not falsas.empty:
        print("\n--- NOTICIAS SOSPECHOSAS DETECTADAS ---")
        print(falsas[["Título", "Votos"]])
    else:
        print("\nNo se encontraron palabras de alarma críticas en este lote de Reddit.")
        
    # Guardar los reportes en archivos CSV listos para tus gráficos
    df_reddit.to_csv("reporte_reddit_limpio.csv", index=False, encoding="utf-8")
    falsas.to_csv("alertas_reddit_falsas.csv", index=False, encoding="utf-8")
    print("\nArchivos CSV exportado ")

else:
    print("Error: No se pudieron extraer datos de la página web.")

Conectando a Reddit 

--- REPORTE DE EXTRACCIÓN ---
Total de noticias evaluadas: 25
Noticias normales validadas: 24
Alertas de información falsa encontradas: 1

--- NOTICIAS SOSPECHOSAS DETECTADAS ---
                  Título  Votos
12  Estafa DAZN: Cuidado    122

Archivos CSV exportado 
